In [1]:
!pip install wandb

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import minmax_scale
from sklearn.model_selection import KFold

import tensorflow as tf

import keras
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Flatten, MaxPool2D, Conv2D
from tensorflow.keras.models import load_model, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, Callback  
from tensorflow.keras.regularizers import L1L2, L1, L2
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

from tqdm import tqdm
import gc

import wandb

import os

In [3]:
os.environ['WANDB_API_KEY'] = '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'
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: andrescubillo07 (andrescubillo07-tecnol-gico-de-costa-rica) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
negative_pairs_path = "../Data_Demo/interactions/txt_interac_CNN/negative_pairs.txt"
positive_pairs_path = "../Data_Demo/interactions/txt_interac_CNN/mirnas_lncrnas_validated_positive_pairs.txt"

positive_pairs = [[line.strip().split(",")[0],line.strip().split(",")[1]] for line in open(positive_pairs_path,"r").readlines()]
negative_pairs = [[line.strip().split(",")[0],line.strip().split(",")[1]] for line in open(negative_pairs_path,"r").readlines()]

all_pairs = positive_pairs + negative_pairs

data = np.load("./data2D.npy")
labels = [1] * len(positive_pairs) + [0] * len(negative_pairs)
labels = to_categorical(labels, num_classes=2)
print(data.shape)

bin_size=32

(45977, 96, 32, 6)


In [4]:
def get_model(dp,n1,n2,n3):
    inx = Input(shape=(bin_size*3, bin_size, 6))
    x = BatchNormalization()(inx)
    x = Conv2D(n1, kernel_size=(3, 3), activation="relu", kernel_regularizer=L1(0.002))(x)
    x = Dropout(dp)(x)
    x = Conv2D(n1, kernel_size=(3, 3), activation="relu", kernel_regularizer=L1(0.002))(x)
    x = Conv2D(n2, kernel_size=(3, 3), activation="relu", kernel_regularizer=L1(0.002))(x)
    x = Conv2D(n2, kernel_size=(3, 3), activation="relu")(x)
    x = Dropout(dp)(x)
    x = Conv2D(n3, kernel_size=(3, 3), activation="relu")(x)
    x = MaxPool2D(pool_size=(2,2))(x)
    x = Flatten()(x)
    x = BatchNormalization()(x)
    x = Dense(256, activation="relu")(x)
    x = Dense(128, activation="relu")(x)
    x = Dense(64, activation="relu")(x)
    x_out = Dense(2, activation="softmax")(x)
    model = Model(inputs=[inx], outputs=[x_out])
    return model

In [5]:
## Inicializa wandb
wandb.init(project="Train-2D-2BinF-4Diccio-2V")

# Define la configuración del sweep
sweep_config = {
    'method': 'grid',
    'metric': {
        'name': 'val_loss',
        'goal': 'minimize'
    },
    'parameters': {
        'dropout': {
            'values': [0.1]
        },
        'menor': {
            'values': [32]
        },
        'medio': {
            'values': [128]
        },
        'alto': {
            'values': [128]
        },
        'tasa_aprend': {
            'values': [0.001]
        },
        'batch': {
            'values': [256]
        }
    }
}

# Inicia el sweep
sweep_id = wandb.sweep(sweep_config, project="Train-2D-2BinF-4Diccio-2V")

# Callback para registrar las metricas de cada epoch
class WandbMetricsLogger(Callback):
    def on_epoch_end(self, epoch, logs=None):
        if logs is not None:
            wandb.log({
                'epoch': epoch,
                'loss': logs.get('loss'),
                'val_loss': logs.get('val_loss'),
                'accuracy': logs.get('accuracy'),
                'val_accuracy': logs.get('val_accuracy'),
                'auc': logs.get('auc'),
                'val_auc': logs.get('val_auc'),
                'precision': logs.get('precision'),
                'val_precision': logs.get('val_precision'),
                'recall': logs.get('recall'),
                'val_recall': logs.get('val_recall'),
                'binary_accuracy': logs.get('binary_accuracy'),
                'val_binary_accuracy': logs.get('val_binary_accuracy')
            })
            
def train():
    wandb.init()
    config = wandb.config
    strategy = tf.distribute.MirroredStrategy()

    global data, labels
    data = np.array(data, dtype=np.float32)
    labels = np.array(labels, dtype=np.float32)

    indices = np.random.permutation(len(data))
    data = data[indices]
    labels = labels[indices]
        
    val_size = int(0.1 * len(data))
    x_final_val = data[:val_size]
    y_final_val = labels[:val_size]

    data = data[val_size:]
    labels = labels[val_size:]
    
    histories = []
    
    split_idx = int(len(data) * 0.8)
    X_train, X_val = data[:split_idx], data[split_idx:]
    y_train, y_val = labels[:split_idx], labels[split_idx:]
    
    with strategy.scope():
        model = get_model(dp=config.dropout, n1=config.menor, n2=config.medio, n3=config.alto)
        adam = Adam(learning_rate=config.tasa_aprend)
        model.compile(optimizer=adam, loss='binary_crossentropy', 
                      metrics=['accuracy', tf.keras.metrics.AUC(), tf.keras.metrics.Precision(), 
                               tf.keras.metrics.Recall(), tf.keras.metrics.BinaryAccuracy()])

    early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                        batch_size=config.batch, shuffle=True, epochs=30,
                        callbacks=[early_stopping, WandbMetricsLogger()])

    histories.append(history)
    
    final_metrics = model.evaluate(x_final_val, y_final_val)

    wandb.log({
        'final_loss': final_metrics[0],
        'final_accuracy': final_metrics[1],
        'final_auc': final_metrics[2],
        'final_precision': final_metrics[3],
        'final_recall': final_metrics[4],
        'final_binary_accuracy': final_metrics[5]
    })
    
    return histories

# Ejecutar el sweep
wandb.agent(sweep_id, function=train)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/andrescubillovillalobos/.netrc.
wandb: Currently logged in as: andrescubillo07 (andrescubillo07-tecnol-gico-de-costa-rica) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Create sweep with ID: mo9kdn9y
Sweep URL: https://wandb.ai/andrescubillo07-tecnol-gico-de-costa-rica/Train-2D-2BinF-4Diccio-2V/sweeps/mo9kdn9y


wandb: Agent Starting Run: tyn1x5tk with config:
wandb: 	alto: 128
wandb: 	batch: 256
wandb: 	dropout: 0.1
wandb: 	medio: 128
wandb: 	menor: 32
wandb: 	tasa_aprend: 0.001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/andrescubillovillalobos/.netrc.


wandb: 
wandb: 🚀 View run decent-thunder-3 at: https://wandb.ai/andrescubillo07-tecnol-gico-de-costa-rica/Train-2D-2BinF-4Diccio-2V/runs/mtw9xm98
wandb: Find logs at: wandb/run-20260617_223432-mtw9xm98/logs


INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:CPU:0',)
Epoch 1/30


...

144/144 ━━━━━━━━━━━━━━━━━━━━ 17s 121ms/step - accuracy: 0.8706 - auc: 0.9427 - binary_accuracy: 0.8706 - loss: 0.3175 - precision: 0.8706 - recall: 0.8706


accuracy,▁▂▂▂▂▃▄▅▇█
auc,▁▂▃▃▄▅▆▆▇█
binary_accuracy,▁▂▂▂▂▃▄▅▇█
epoch,▁▂▃▃▄▅▆▆▇█
final_accuracy,▁
final_auc,▁
final_binary_accuracy,▁
final_loss,▁
final_precision,▁
final_recall,▁
+9,...


wandb: Sweep Agent: Waiting for job.
wandb: Sweep Agent: Exiting.


Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x16c2cfc50>> (for post_run_cell), with arguments args (<ExecutionResult object at 16bc33210, execution_count=5 error_before_exec=None error_in_exec=None info=<ExecutionInfo object at 16bc32990, raw_cell="## Inicializa wandb
wandb.init(project="Train-2D-2.." transformed_cell="## Inicializa wandb
wandb.init(project="Train-2D-2.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/Users/andrescubillovillalobos/Documents/CompGen_Inc%20%28Reorganizado%29/CNN_CompGenInc/CNN_CompGenInc.ipynb#W5sZmlsZQ%3D%3D> result=None>,),kwargs {}:


AlreadyJoinedError: Cannot schedule tasks after join(). Did you call wandb.teardown()?

In [10]:
from tensorflow.keras.models import load_model
model = load_model('./models/CNN_CV_Model.keras')

In [12]:
from tensorflow.keras.models import load_model
import numpy as np

model = load_model('./models/CNN_CV_Model.keras')
data = np.load('./data2D.npy')

# Tomar una muestra para predecir
sample = data[:100]
predicted = model.predict(sample)
print(predicted)

4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 137ms/step
[[6.28711730e-02 9.37128901e-01]
 [7.96915293e-02 9.20308471e-01]
 [5.58295622e-02 9.44170415e-01]
 [9.99968112e-01 3.19381652e-05]
 [7.09927175e-03 9.92900729e-01]
 [1.97451264e-01 8.02548766e-01]
 [8.23258162e-02 9.17674124e-01]
 [4.84284721e-02 9.51571465e-01]
 [6.99031800e-02 9.30096805e-01]
 [1.36507854e-01 8.63492131e-01]
 [9.98489335e-02 9.00151074e-01]
 [1.25171870e-01 8.74828100e-01]
 [8.08791593e-02 9.19120789e-01]
 [8.76134560e-02 9.12386537e-01]
 [6.11397885e-02 9.38860178e-01]
 [5.20096496e-02 9.47990298e-01]
 [7.21580982e-02 9.27841842e-01]
 [9.26434398e-02 9.07356620e-01]
 [9.18671936e-02 9.08132732e-01]
 [1.93614393e-01 8.06385696e-01]
 [9.89067927e-02 9.01093125e-01]
 [3.44448239e-02 9.65555131e-01]
 [1.17722146e-01 8.82277906e-01]
 [4.91633713e-02 9.50836658e-01]
 [2.63906866e-02 9.73609269e-01]
 [4.50565927e-02 9.54943478e-01]
 [1.02622196e-01 8.97377789e-01]
 [8.98920447e-02 9.10107970e-01]
 [9.12332758e-02 9.08766806e-01]
 [6.